In [1]:
# Kill all processes on the GPU
!fuser -v /dev/nvidia* -k

In [2]:
# Check GPU status
!nvidia-smis

/bin/bash: line 1: nvidia-smis: command not found


In [3]:
# For Qwen3-VL
# %%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    %pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    %pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    %pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
%pip install transformers==4.57.1
%pip install --no-deps trl==0.22.2

# Configuration

In [27]:
SEED = 42

# Model configuration
SFT_LORA_ID = 'alxxtexxr/Qwen3-VL-8B-Thinking-VCB8K-SFT-QLoRA-v20260421190643'

# Data configuration
RL_DATA_ID = 'alxxtexxr/ViRL1.25K'

# Training configuration
MINI_BATCH_SIZE = 2
GRAD_ACC_STEPS = 4
NUM_EPOCHS = 1
WARMUP_STEPS = 5
LR = 5e-6

In [13]:
from datetime import datetime

# Resume training configuration
RESUME_MODEL_ID = None
resume_from_checkpoint = bool(RESUME_MODEL_ID)
if resume_from_checkpoint:
    # hub_model_id = RESUME_MODEL_ID
    # project_name = hub_model_id.split('/')[-1]
    # model_name = project_name

    # from huggingface_hub import snapshot_download
    # snapshot_download(repo_id=hub_model_id, local_dir=model_name)
    
    raise NotImplementedError("Resuming from checkpoint is not implemented yet!")
else:
    model_name = SFT_LORA_ID
    project_name = f'{model_name.split("/")[-1].split('-SFT-QLoRA-v')[0]}-GRPO-QLoRA-v{datetime.now().strftime("%Y%m%d%H%M%S")}'
    hub_model_id = f'alxxtexxr/{project_name}'
print("Resume from checkpoint:", resume_from_checkpoint)
print("Model name:", model_name)
print("Project name:", project_name)
print("Hugging Face model ID:", hub_model_id)

Resume from checkpoint: False
Model name: alxxtexxr/Qwen3-VL-8B-Thinking-VCB8K-SFT-QLoRA-v20260421190643
Project name: Qwen3-VL-8B-Thinking-VCB8K-GRPO-QLoRA-v20260422184741
Hugging Face model ID: alxxtexxr/Qwen3-VL-8B-Thinking-VCB8K-GRPO-QLoRA-v20260422184741


In [ ]:
import os
from unsloth import FastVisionModel
import random
import numpy as np
import torch
import transformers

def set_seed(seed, verbose=True):
    # Set random seed for os
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':16:8'  # CUDA deterministic behavior (11.2+)

    # Set random seed for NumPy
    np.random.seed(seed)

    # Set random seed for Torch
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)           # If using multi-GPU
    # torch.use_deterministic_algorithms(True) # ERROR!
    torch.backends.cudnn.deterministic = True  # Ensures deterministic results
    torch.backends.cudnn.benchmark = False     # Avoids non-deterministic algorithms

    # Set random seed for Transformers
    transformers.set_seed(seed)

    # Set random seed for sklearn and Python's own random module
    random.seed(seed) 
    
    if verbose:
        print("Set random seed:", seed)

set_seed(SEED)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Random seed: 42


In [15]:
# Fix TorchDynamo issue
os.environ['TORCHDYNAMO_DISABLE'] = '1'
os.environ['UNSLOTH_DISABLE_COMPILE'] = '1'
os.environ['UNSLOTH_DISABLE_FUSED_KERNELS'] = '1'

# Fix Unsloth issue
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = ''

# Model

In [16]:
model, tokenizer = FastVisionModel.from_pretrained(
    model_name = SFT_LORA_ID,
    load_in_4bit = True, # Use 4bit to reduce memory use. False for 16-bit LoRA.
    use_gradient_checkpointing = 'unsloth', # True or 'unsloth' for long context
    # dtype = torch.float16, # Force FP16
)

==((====))==  Unsloth 2026.4.7: Fast Qwen3_Vl patching. Transformers: 4.57.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

# Data

In [17]:
from datasets import load_dataset

dataset = load_dataset(RL_DATA_ID)
print(dataset)

README.md:   0%|          | 0.00/830 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/47.8M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/3.98M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/6.67M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/125 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/125 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['question', 'answer', 'PassRate_32BTrained', 'PassRate_7BBase', 'category', 'source', 'qid', 'image_paths', 'image'],
        num_rows: 1000
    })
    validation: Dataset({
        features: ['question', 'answer', 'PassRate_32BTrained', 'PassRate_7BBase', 'category', 'source', 'qid', 'image_paths', 'image'],
        num_rows: 125
    })
    test: Dataset({
        features: ['question', 'answer', 'PassRate_32BTrained', 'PassRate_7BBase', 'category', 'source', 'qid', 'image_paths', 'image'],
        num_rows: 125
    })
})


In [18]:
train_set = dataset['train']
val_set = dataset['validation']

print("Train set:")
print(train_set)
print("\nValidation set:")
print(val_set)

Train set:
Dataset({
    features: ['question', 'answer', 'PassRate_32BTrained', 'PassRate_7BBase', 'category', 'source', 'qid', 'image_paths', 'image'],
    num_rows: 1000
})

Validation set:
Dataset({
    features: ['question', 'answer', 'PassRate_32BTrained', 'PassRate_7BBase', 'category', 'source', 'qid', 'image_paths', 'image'],
    num_rows: 125
})


In [19]:
# Resize to (512, 512)
def resize_images(example):
    image = example["image"]
    image = image.resize((512, 512))
    example["decoded_image"] = image
    return example
train_set = train_set.map(resize_images)
val_set = val_set.map(resize_images)

# Then convert to RGB
def convert_to_rgb(example):
    image = example["decoded_image"]
    if image.mode != "RGB":
        image = image.convert("RGB")
    example["decoded_image"] = image
    return example
train_set = train_set.map(convert_to_rgb)
val_set = val_set.map(convert_to_rgb)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/125 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/125 [00:00<?, ? examples/s]

In [20]:
def clean_question(example):
    question = example["question"]
    question = question.replace("<image>", "")    # Remove "<image>" from the question text
    question = " ".join(question.strip().split()) # Remove extra whitespace
    example["question"] = question
    return example

train_set = train_set.map(clean_question)
val_set = val_set.map(clean_question)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/125 [00:00<?, ? examples/s]

In [21]:
# Define the delimiter variables for clarity and easy modification
THINK_START = "<think>"
THINK_END = "</think>"
VTHINK_START = "<vthink>"
VTHINK_END = "</vthink>"
ANSWER_START = "<answer>"
ANSWER_END = "</answer>"

def make_conversation(example):
    # Define placeholder constants if they are not defined globally
    # The user's text prompt
    text_content = (
        f"{example['question']}\n\n"
        f"Please first reason step by step using visual representations between {VTHINK_START} and {VTHINK_END}, "
        f"then reason step by step in natural language between {THINK_START} and {THINK_END}, "
        "and finally provide your final answer in LaTeX using \\boxed{} "
        f"between {ANSWER_START} and {ANSWER_END}"
    )

    # Construct the prompt in the desired multi-modal format
    prompt = [
        {
            "role": "user",
            "content": [
                {"type": "image"},  # Placeholder for the image
                {"type": "text", "text": text_content},  # The text part of the prompt
            ],
        },
    ]
    # The actual image data is kept separate for the processor
    return {"prompt": prompt, "image": example["decoded_image"], "answer": example["answer"]}

train_dataset = train_set.map(make_conversation)
val_dataset = val_set.map(make_conversation)

# We're reformatting dataset like this because decoded_images are the actual images
# The "image": example["decoded_image"] does not properly format the dataset correctly

# 1. Remove the original 'image' column
train_dataset = train_dataset.remove_columns("image")
val_dataset = val_dataset.remove_columns("image")

# 2. Rename 'decoded_image' to 'image'
train_dataset = train_dataset.rename_column("decoded_image", "image")
val_dataset = val_dataset.rename_column("decoded_image", "image")

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/125 [00:00<?, ? examples/s]

In [22]:
image = train_dataset[100]["image"]
prompt = train_dataset[100]["prompt"]

inputs = tokenizer(
    image,
    prompt,
    add_special_tokens = False,
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = False)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 1024,
                   use_cache = True, temperature = 1.0, min_p = 0.1)

<|im_start|>user
<|vision_start|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|ima

# Reward Functions

In [25]:
# Reward functions
import re

def formatting_reward_func(completions,**kwargs):
    import re
    thinking_pattern = f'{THINK_START}(.*?){THINK_END}'
    vthinking_pattern = f'{VTHINK_START}(.*?){VTHINK_END}'
    answer_pattern = f'{ANSWER_START}(.*?){ANSWER_END}'

    scores = []
    for completion in completions:
        if isinstance(completion, list):
            completion = completion[0]["content"] if completion else ""
        score = 0
        thinking_matches = re.findall(thinking_pattern, completion, re.DOTALL)
        vthinking_matches = re.findall(vthinking_pattern, completion, re.DOTALL)
        answer_matches = re.findall(answer_pattern, completion, re.DOTALL)
        if len(thinking_matches) == 1:
            score += (1/3)
        if len(vthinking_matches) == 1:
            score += (1/3)
        if len(answer_matches) == 1:
            score += (1/3)

        # Fix up addCriterion issues
        # See https://unsloth.ai/docs/new/vision-reinforcement-learning-vlm-rl#qwen-2.5-vl-vision-rl-issues-and-quirks
        # Penalize on excessive addCriterion and newlines
        if len(completion) != 0:
            removal = completion.replace("addCriterion", "").replace("\n", "")
            if (len(completion)-len(removal))/len(completion) >= 0.5:
                score -= 1.0

        scores.append(score)
    return scores


def correctness_reward_func(prompts, completions, answer, **kwargs) -> list[float]:
    answer_pattern = f'{ANSWER_START}(.*?){ANSWER_END}'

    completions = [(c[0]["content"] if c else "") if isinstance(c, list) else c for c in completions]
    responses = [re.findall(answer_pattern, completion, re.DOTALL) for completion in completions]
    q = prompts[0]
    print('-'*20, f"Question:\n{q}", f"\nAnswer:\n{answer[0]}", f"\nResponse:{completions[0]}")
    return [
        1.0 if len(r)==1 and a == r[0].replace('\n','') else 0.0
        for r, a in zip(responses, answer)
    ]

# Training

In [28]:
import math
from trl import GRPOConfig, GRPOTrainer

max_steps = math.ceil(len(train_dataset) / (MINI_BATCH_SIZE * GRAD_ACC_STEPS)) * NUM_EPOCHS
print("Calculated max steps:", max_steps)

training_args = GRPOConfig(
    # Training arguments
    seed = SEED,
    
    per_device_train_batch_size = MINI_BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACC_STEPS,
    num_generations = 4, # Decrease if out of memory
    
    max_prompt_length = 1024,
    max_completion_length = 1024,
    
    # max_steps = 50, # For testing
    max_steps = max_steps,
    # num_train_epochs = NUM_EPOCHS,
    # warmup_ratio=0.05,
    warmup_steps = WARMUP_STEPS,
    learning_rate = LR,
    lr_scheduler_type = "cosine",
    optim = "adamw_8bit",
    max_grad_norm = 0.1,
    weight_decay = 0.01,
    adam_beta1 = 0.9,
    adam_beta2 = 0.99,
    
    # Logging arguments
    logging_strategy='steps',
    logging_steps = 1,
    log_completions = False,
    report_to=['tensorboard', 'wandb'],
    
    # Saving arguments
    save_strategy='steps',
    # save_steps=1, # For testing
    save_steps=20,
    # save_total_limit=5, # 1 best + 4 recent checkpoints. Warning: It doesn't work
    
    # With load_best_model_at_end=True, your save_strategy will be ignored and default to eval_strategy.
    # So you will find one checkpoint at the end of each epoch.
    # https://discuss.huggingface.co/t/trainer-not-saving-after-save-steps/5464
    # load_best_model_at_end=True, 

    output_dir=project_name,
    hub_model_id=hub_model_id,
    push_to_hub=True,

    hub_strategy='all_checkpoints',
    hub_always_push=True,

    # Below enables GSPO:
    importance_sampling_level = "sequence",
    mask_truncated_completions = False,
    loss_type = "dr_grpo",
)

Calculated max steps: 125


In [ ]:
trainer = GRPOTrainer(
    model = model,
    args = training_args,
    # Pass the processor to handle multimodal inputs
    processing_class = tokenizer,
    reward_funcs = [
        formatting_reward_func,
        correctness_reward_func,
    ],
    train_dataset = train_dataset,
)

trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,000 | Num Epochs = 1 | Total steps = 125
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 43,646,976 of 8,875,708,656 (0.49% trained)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: alimtegar to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/
`generation_config` default values have been modified to match model-specific defaults: {'top_p': 0.95}. If this is not desired, please set these values explicitly.


-------------------- Question:
[{'role': 'user', 'content': [{'type': 'image', 'text': None}, {'type': 'text', 'text': "In the figure, two equilateral triangles ABD and CBD each have side lengths of 1. Triangle ABD is translated to the right along the AC direction into position A'B'D', creating a shaded area. What is the perimeter of this shaded area? The above problem is with the following images:\n\nPlease first reason step by step using visual representations between <vthink> and </vthink>, then reason step by step in natural language between <think> and </think>, and finally provide your final answer in LaTeX using \\boxed{} between <answer> and </answer>"}]}] 
Answer:
\boxed{2} 
Response:</think>

<think><|vtok_409|><|vtok_2012|><|vtok_314|><|vtok_314|><|vtok_1719|><|vtok_2597|><|vtok_6115|><|vtok_6293|><|vtok_5846|><|vtok_2301|><|vtok_80|><|vtok_1024|><|vtok_4247|><|vtok_7451|><|vtok_3842|><|vtok_8|><|vtok_302|><|vtok_1153|><|vtok_3099|><|vtok_7531|><|vtok_6056|><|vtok_3663|><|vt

Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / formatting_reward_func / mean,rewards / formatting_reward_func / std,rewards / correctness_reward_func / mean,rewards / correctness_reward_func / std
1,0.107500,0.208333,0.315495,861.875000,375.000000,1024.000000,0.750000,375.500000,375.000000,376.000000,3.082181,0.083333,0.154303,0.125000,0.353553
2,0.149600,0.166667,0.166667,764.125000,231.000000,1024.000000,0.500000,504.250000,231.000000,831.000000,1.935426,0.166667,0.178174,0.000000,0.000000
3,0.100300,0.291667,0.372008,883.000000,384.000000,1024.000000,0.750000,460.000000,384.000000,536.000000,5.092718,0.166667,0.178174,0.125000,0.353553
4,0.004200,0.125000,0.159571,1024.000000,1024.000000,1024.000000,1.000000,0.000000,0.000000,0.000000,4.241118,0.125000,0.248008,0.000000,0.000000
5,-0.027400,0.041667,0.083333,958.125000,505.000000,1024.000000,0.750000,760.500000,505.000000,1016.000000,4.221231,0.041667,0.117851,0.000000,0.000000
6,0.036300,0.166667,0.242905,897.625000,13.000000,1024.000000,0.875000,13.000000,13.000000,13.000000,4.408601,0.166667,0.251976,0.000000,0.000000
7,0.095200,0.166667,0.166667,835.875000,20.000000,1024.000000,0.625000,522.333374,20.000000,952.000000,3.696917,0.166667,0.178174,0.000000,0.000000
8,0.253600,0.083333,0.166667,853.125000,42.000000,1024.000000,0.750000,340.500000,42.000000,639.000000,3.855107,0.083333,0.154303,0.000000,0.000000
9,0.220200,0.458333,0.439817,544.625000,14.000000,1024.000000,0.375000,257.000000,14.000000,911.000000,1.213558,0.208333,0.172516,0.250000,0.462910
10,0.078300,0.125000,0.179558,935.125000,557.000000,1024.000000,0.750000,668.500000,557.000000,780.000000,3.480728,0.125000,0.172516,0.000000,0.000000


-------------------- Question:
[{'role': 'user', 'content': [{'type': 'image', 'text': None}, {'type': 'text', 'text': "On the 90th anniversary of the founding of the Communist Party of China, the County Education Bureau held a 'Red Songs Resound in China' activity. A 'Red Songs' singing competition was held in June. The organizing committee stipulates: any participant's score x satisfies: 60 ≤ x < 100. After the competition, the scores of all participants were organized as shown in the following table: According to the information provided in the table, n = ___.\n\nPlease first reason step by step using visual representations between <vthink> and </vthink>, then reason step by step in natural language between <think> and </think>, and finally provide your final answer in LaTeX using \\boxed{} between <answer> and </answer>"}]}] 
Answer:
\boxed{0.25} 
Response: user
<think><|vtok_325|> bar chart<|vtok_6692|><|vtok_4817|><|vtok_279|><|vtok_7604|><|vtok_6806|><|vtok_3606|><|vtok_5608|><|

# Inference

In [16]:
FastVisionModel.for_inference(model) # Enable for inference!

image = None
instruction = "Several slices of bread with side salads on a table."

messages = [
    {"role": "user", "content": [
        # {"type": "image"},
        {"type": "text", "text": instruction}
    ]}
]
input_text = tokenizer.apply_chat_template(messages, add_generation_prompt = True)
inputs = tokenizer(
    image,
    input_text,
    add_special_tokens = False,
    return_tensors = "pt",
).to("cuda")

In [17]:
from transformers import TextStreamer

text_streamer = TextStreamer(tokenizer, skip_prompt = False)
outputs = model.generate(**inputs, 
                   streamer = text_streamer, 
                   max_new_tokens = 512,
                   use_cache = True, temperature = 1.5, min_p = 0.1)

<|im_start|>user
Several slices of bread with side salads on a table.<|im_end|>
<|im_start|>assistant
<|vtok_5424|><|vtok_1834|><|vtok_5261|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><|vtok_5569|><